# 4.7 一次訓練多個位置怎麼對齊答案？

# 第 4 章：第一個 Dense Transformer 語言模型

前置：第3章的因果 attention。完整可讀實作在 tiny_perceptron/model.py；預設只用 learned position、LayerNorm、Dense FFN，還沒有 FlashAttention 或 MoE。




In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：像原稿旁加修訂，而不是撕掉原稿**

y=x+f(x)，原資料和修訂都有路徑。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/residual.svg")))

**先想一想：**模型內shift加資料外shift，會猜到幾格後的字？

整段一次forward時，每個位置各猜自己的下一字。相同因果mask允許平行算所有訓練位置；生成時才必須逐步加入新字。

**把直覺寫成數學：**原序列 [a,b,c,d] → X=[a,b,c]，Y=[b,c,d]，logits 的每格對應同位置的Y。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
from tiny_perceptron.data import shifted
from tiny_perceptron.model import TinyLM, ModelConfig, masked_loss

x, y = shifted([1, 2, 3, 4])
model = TinyLM(ModelConfig(vocab_size=5, width=8))
logits = model(x[None])["logits"]
loss = masked_loss(logits, y[None])
print(list(zip(x.tolist(), y.tolist())), loss.item())
loss.backward()

**如何讀結果：**loss會把每個有效位置平均；shift只做一次，TinyLM本身不改labels。

**只改一件事：**只故意把Y再shift一次，說出shape或語意哪裡錯。
